In [1]:
# TODO this is temporary until we create a subpackage!
import sys
sys.path.append("../../.."); sys.path.append("../..") ;sys.path.append("..")
import library

In [2]:
# Imports for calibration TODO to be moved
import mergedeep
import copy

# General Imports
import yaml
import os
thisfiledir = os.getcwd()
expdir = '/home/adminlwe/Documents/lwe-opu/experiment'
savedir = expdir + '/data/model_training'
import time
import numpy as np
import pandas as pd
import ipywidgets
import cv2
from tqdm import tqdm
import pickle
from shutil import copyfile
from datetime import datetime
import tracemalloc

import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from IPython.display import display, clear_output
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import curve_fit
from scipy.signal import convolve2d, fftconvolve
from scipy.special import jv
from scipy.ndimage import gaussian_filter

# Pytorch imports
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

import torch.optim as optim
from torch.optim import Adam, RMSprop
from torchvision.datasets import MNIST, FashionMNIST
from torchvision.transforms import Compose, ToTensor, Normalize, Lambda
from torch.utils.data import DataLoader, default_collate

# Custom modules and NN
from library.software.extra import ComplexLayerNorm, ComplexAvgPool2d, CPU2GPU, DebugFigure, Repeat
from library.software.layers import FFConv, FFLinear, FFElmwise, FFPhaseElmwise
from library.software.layers import Screen, Camera, Mask

from library.software.architectures import NormalNet, FFNet
from library.software.dataloaders import MNIST_loaders
from op_torch import img_nav

# Camera imports
from library.hardware.PylonCamera import PylonCamera
from library.hardware import DispUtils

# Display imports
from library.hardware import DispUtils
from library.hardware.DisplayGL import DisplayGL
from library.hardware.PylonCamera import PylonCamera
from library.hardware.DeviceManager import DeviceManager
from library.software import ExpModels

# Select GPU as device if possible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [3]:
def bgr8_to_jpeg(frame, quality=75):
    return bytes(cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, quality])[1])

def update_widget(img_widget, value):
    img_widget.value = bgr8_to_jpeg(value)

img_widget = ipywidgets.Image(format='jpeg', value=bgr8_to_jpeg(np.zeros((1,1))), width=400, height=300)

### Data & Batch size instantiation

In [4]:
zoom = 4
img_size = (28*zoom, 28*zoom) ## image size on the Screen sent to uD
slm_size = (28*zoom, 28*zoom) ## size of the mask
pat_pad = (42, 50) ## padding next to each image, must be even numbers
cam_pad = (6, 6) ## Padding to add when capturing imagelet with the camera
batch_layout = (4,4) ## nb of images
batch_stacks = 1 ## nb images in the queue for displaying, too high: increase batch size, risk of filling memory + bad learning, too low: slower
shape_labeled = img_size
thres_lst = [[45, 6], [25, 3]]
kern_lst = [[5, 3], [5, 3]]
Niter_slm = 6

### Hardware initialisation

In [5]:
uD, slm, camD, cam0, dev_mgmt = ExpModels.initiate_devices(expD = 5000, exp0 = 5000)

Camera devices ID found: ['40308273', '40627052']
Camera devices ID found: ['40308273', '40627052']
Using GLFW: 3.4.0 Wayland X11 GLX Null EGL OSMesa monotonic shared


### Single calibration

In [6]:
camD._camera.ExposureTime.Value, cam0._camera.ExposureTime.Value = 6000, 6000
cal_dict, slm_xy_offsets = ExpModels.calibration(shape_labeled, pat_pad, cam_pad, batch_layout, batch_stacks, Niter_slm, thres_lst=thres_lst, kern_lst=kern_lst)

Calibrating screen 0...
Calibrating screen 1 ...
{'slm-in_size': (112, 112), 'slm-out#0_size': (136, 138), 'slm-out#1_size': (136, 138), 'padding': (42, 50), 'batch': (4, 4), 'uD-in_size': (112, 112), 'uD-out#0_size': (120, 122), 'uD-out#1_size': (134, 136)}


In [ ]:
if False:
    filename = f'zoom={zoom}'
    os.chdir(thisfiledir +'/calibration')
    file = open('calibration_' + filename, 'wb')
    pickle.dump([cal_dict, slm_xy_offsets], file)
    file = open('data_' + filename, 'wb')
    pickle.dump([zoom, img_size, slm_size, pat_pad, cam_pad, batch_layout, batch_stacks, shape_labeled, Npix], file)
    file.close() ; os.chdir(thisfiledir)

## Sequential calibration

In [13]:
camD._camera.ExposureTime.Value, cam0._camera.ExposureTime.Value = 6000, 6000
thres_lst = [[45, 6], [35, 3]]
kern_lst = [[7, 3], [7, 3]]
zoom_list = [4, 6 ]
layout_list = [batch_layout, batch_layout, batch_layout, batch_layout] #[(8, 8), (4,8), (4,8), (4, 4)]
for iz,zoom in enumerate(zoom_list):
    img_size = (28*zoom, 28*zoom) ## image size on the Screen sent to uD
    layout = layout_list[iz]
    shape_labeled = img_size
    filename = f'zoom={zoom}'
    cal_dict, slm_xy_offsets = ExpModels.calibration(shape_labeled, pat_pad, cam_pad, batch_layout, batch_stacks, Niter_slm, thres_lst=thres_lst, kern_lst=kern_lst)
    Nmux = batch_layout[0]*batch_layout[1]
    (Npix_x, Npix_y) = cal_dict['info']['uD-out#0_size'] #the output size
    Npix = (Npix_x, Npix_y)
    Npixtot = Npix_x * Npix_y
    Nin = shape_labeled[0]*shape_labeled[1]
    os.chdir('calibration')
    file = open('calibration_' + filename, 'wb')
    pickle.dump([cal_dict, slm_xy_offsets], file)
    file = open('data_' + filename, 'wb')
    pickle.dump([zoom, img_size, slm_size, pat_pad, cam_pad, batch_layout, batch_stacks, shape_labeled, Npix], file)
    file.close() ; os.chdir(thisfiledir)

Calibrating screen 0...
Calibrating screen 1 ...
{'slm-in_size': (112, 112), 'slm-out#0_size': (136, 138), 'slm-out#1_size': (136, 138), 'padding': (42, 50), 'batch': (4, 4), 'uD-in_size': (112, 112), 'uD-out#0_size': (120, 122), 'uD-out#1_size': (134, 134)}


Calibrating screen 0...
Calibrating screen 1 ...
{'slm-in_size': (168, 168), 'slm-out#0_size': (192, 194), 'slm-out#1_size': (196, 198), 'padding': (42, 50), 'batch': (4, 4), 'uD-in_size': (168, 168), 'uD-out#0_size': (176, 178), 'uD-out#1_size': (194, 196)}


## Caracterization

In [14]:
train_wid_all = ipywidgets.Image(format='jpeg', value=bgr8_to_jpeg(np.zeros((1,1))), width=1000, height=550)
train_wid_one = ipywidgets.Image(format='jpeg', value=bgr8_to_jpeg(np.zeros((1,1))), width=1000, height=550)

uD_layer = Screen(img_shape=(*batch_layout, *shape_labeled), batch_stacks=batch_stacks, xy_offsets=None, screen_dev=uD, cal_dict=cal_dict, device=device)
SLM_layer = Mask(img_shape=(*batch_layout, *shape_labeled), batch_stacks=batch_stacks, xy_offsets=slm_xy_offsets, screen_dev=slm, cal_dict=cal_dict, device=device)
CAM_layer = Camera(img_shape=(*batch_layout, *shape_labeled), batch_stacks=batch_stacks, device=device, cam_dev=camD, cal_dict=cal_dict, cal_key='uD-out#0', #update_widget=None)
                         update_widget=[lambda x: update_widget(train_wid_all, x), lambda x: update_widget(train_wid_one, x)])
                        #update_widget=None)

def dummy_model(imgs_uD, imgs_SLM):
    SLM_layer._weight.data = imgs_SLM
    uD_layer.forward(imgs_uD)
    SLM_layer.forward(None)
    out=CAM_layer.forward(None)
    return out#.reshape( CAM_layer._nh * CAM_layer._nw, CAM_layer._batch_stacks, CAM_layer._h, CAM_layer._w) 
    #return cam_dev.capture_image(30, sync_frame=False)

In [11]:
#resetting
camD._camera.ExposureTime.Value = 2000
#cam_dev.reset_screen_trigger(sync_time=True)

#### Loading

In [17]:
os.chdir(thisfiledir +'/calibration')
file = open('calibration_' + filename, 'rb')
[cal_dict, slm_xy_offsets] = pickle.load(file); file.close()
file = open('data_' + filename, 'rb')
[zoom, img_size, slm_size, pat_pad, cam_pad, batch_layout, batch_stacks, shape_labeled, Npix] = pickle.load(file)
file.close() ; os.chdir(thisfiledir)

#### Basic examples to play with

In [15]:
def generate_flat_slm_mask(shape_labeled):
    '''
    Input:  shape_labeled: [Lx,Ly] vector
            n_repeat: nb of px for the same value
    Output: slm_stack with the same random mask of shape [batch_stacks, Nmux, Lx, Ly]     
    '''
    random = torch.from_numpy(np.zeros([shape_labeled[0],shape_labeled[1]]))
    slm_stack = torch.stack([ torch.stack([ random for i in range(batch_layout[0]*batch_layout[1])])  for i in range(batch_stacks)]).to(torch.device("cpu:0"))
    return slm_stack

def generate_random_slm_mask(shape_labeled, n_repeat=2):
    '''
    Input:  shape_labeled: [lx,ly] vector
            n_repeat: nb of px for the same value
    Output: slm_stack with the same random mask of shape [batch_stacks, Nx, Ny, Lx, Ly]     
    '''
    random = torch.from_numpy(np.random.uniform(size=[shape_labeled[0]//n_repeat+1,shape_labeled[1]//n_repeat+1]))
    random = torch.repeat_interleave(random, n_repeat, dim=-1); random = torch.repeat_interleave(random, n_repeat, dim=-2)
    slm_stack = torch.stack([torch.stack([ random for i in range(batch_layout[0]*batch_layout[1])])  for i in range(batch_stacks)]).to(torch.device("cpu:0"))
    return slm_stack[:,:, :shape_labeled[0], :shape_labeled[1]]

In [16]:
m1 = generate_random_slm_mask(shape_labeled, n_repeat=1)
m2 = generate_random_slm_mask(shape_labeled, n_repeat=2)
m4 = generate_random_slm_mask(shape_labeled, n_repeat=4)
m6 = generate_random_slm_mask(shape_labeled, n_repeat=6)
m8 = generate_random_slm_mask(shape_labeled, n_repeat=8)
m_flat = generate_flat_slm_mask(shape_labeled)

In [17]:
train_wid = ipywidgets.HBox([train_wid_all, train_wid_one]) ;display(train_wid)

In [18]:
camD._camera.ExposureTime.Value=1500
im2 = dummy_model(torch.ones([batch_stacks, Nmux, *shape_labeled]),m2).cpu().detach().numpy()
im1 = dummy_model(torch.ones([batch_stacks, Nmux, *shape_labeled]),m1).cpu().detach().numpy()
im4 = dummy_model(torch.ones([batch_stacks, Nmux, *shape_labeled]),m4).cpu().detach().numpy()
im6 = dummy_model(torch.ones([batch_stacks, Nmux, *shape_labeled]),m6).cpu().detach().numpy()

In [20]:
camD._camera.ExposureTime.Value = 6000
imblank = dummy_model(torch.ones([batch_stacks, Nmux, *shape_labeled]),m_flat).cpu().detach().numpy()

In [ ]:
def plot_stack(stack, cmap='hot'):
    fig, axs = plt.subplots(nrows=batch_layout[0], ncols=batch_layout[1], figsize=(batch_layout[1]*(zoom-1),batch_layout[0]*(zoom-1)))
    plt.axis('off')
    for i in range(batch_layout[0]):
        for j in range(batch_layout[1]):
            if stack.ndim==4:
                im_ij = stack[i*batch_layout[0]+j,0,::]
                axs[i,j].matshow(im_ij/np.max(im_ij),cmap=cmap)
            elif stack.ndim==3:
                im_ij = stack[i*batch_layout[0]+j,::]
                axs[i,j].matshow(im_ij/np.max(im_ij),cmap=cmap)
            axs[i,j].set_yticklabels([])  ; axs[i,j].set_xticklabels([])
    return fig, axs

In [ ]:
def plot_avg_stack(stack, cmap='hot'):
    im_mean = stack.mean(axis=(0,1))
    plt.matshow(im_mean, cmap=cmap); plt.colorbar()

In [ ]:
plot_avg_stack(im1)

In [ ]:
fig, axs = plot_stack(im2)

In [ ]:
fig,axs=plot_stack(im4)

## PSF measurements

In [24]:
# square of increasing sizes to measure the PSF
custom_PSF_loader = [] #iterable as a loader 
x_mux = []
Npsf = 20
for mes in range(Npsf):
    im0 = np.zeros(shape_labeled)
    im0[shape_labeled[0]//2, shape_labeled[1]//2]=1
    k = np.ones((2*mes,2*mes))
    obj = convolve2d(im0,k,mode='same').astype(np.uint8)
    obj = torch.as_tensor(obj)
    obj_list = torch.stack([obj for n in range(Nmux)])
    obj = torch.stack([obj_list for n in range(batch_stacks)])
    y = mes*np.ones((Nmux))
    custom_PSF_loader.append((obj,y))
#plt.matshow(custom_PSF_loader[1][0].cpu().detach().numpy()[3,0,::]);plt.colorbar()

In [25]:
PSF_l=[]
for im_psf, y_psf in custom_PSF_loader:
    PSF_l.append(dummy_model(im_psf, m_flat))

In [ ]:
cwd = os.getcwd()
os.chdir(cwd +'/data/TM')
file = open(f'PSF_l{datetime.now().strftime("%y_%m_%d_%H:%M")}', 'wb')
pickle.dump(PSF_l, file)
file.close(); os.chdir(cwd)

In [ ]:
plt.matshow(PSF_l[-1].cpu().detach().numpy()[55,0,::]);plt.colorbar()

In [ ]:
plt.matshow(PSF_l[1].cpu().detach().numpy()[55,0,::]-PSF_l[0].cpu().detach().numpy()[55,0,::]);plt.colorbar()

In [ ]:
i_psf = 3
PSF_exp = np.mean(PSF_l[i_psf].cpu().detach().numpy(), axis=0)
#PSF_exp = PSF_exp/np.max(PSF_exp)
plt.matshow(PSF_exp[0,::], cmap='hot') ; plt.colorbar() #Mean PSF
plt.title(f'Experimental Mean PSF')

In [ ]:
PSFxy = PSF_l[i_psf].cpu().detach().numpy()
PSFxy = [[PSFxy[imux+Nmux*i,0,::] for imux in range(Nmux)] for i in range(batch_stacks)] # transfo inv. du reshape de CAM_layer
PSFxy = np.asarray(PSFxy).reshape(batch_stacks, *batch_layout, *Npix)
PSFxy_0 = PSF_l[0].cpu().detach().numpy() #ref measurement
PSFxy_0 = [[PSFxy_0[imux+Nmux*i,0,::] for imux in range(Nmux)] for i in range(batch_stacks)]
PSFxy_0 = np.asarray(PSFxy_0).reshape(batch_stacks, *batch_layout, *Npix)
PSFxy = PSFxy-PSFxy_0
#for i in range(8):
#    plt.matshow(PSFxy[i,10,0,::])

In [ ]:
fig, axs = plt.subplots(nrows=batch_layout[0], ncols=batch_layout[1], figsize=(15,15))
for i_x in range(batch_layout[0]):
    for i_y in range(batch_layout[1]):
        axs[i_x, i_y].matshow(np.sum(PSFxy[:,i_x,i_y,::],axis=0), cmap='hot')
        axs[i_x, i_y].set_yticklabels([]); axs[i_x,i_y].set_xticklabels([]) ; plt.axis('off')